# 150 — Align USLaunchReport video, audio, and BCHH waveforms

This workbench treats **BCHH as the reference clock**.  The USLaunchReport YouTube movie has no independent absolute UTC; its video and embedded audio are therefore aligned *to* BCHH, not used to redefine BCHH timing.

The workflow is deliberately two-stage:

1. load the event catalog, rank the 10 strongest events, add any of the four original named events that are not already represented, and manually refine the observed arrival time on each BCHH pressure channel;
2. reduce those BCHH arrival picks to a source time using the selected acoustic speed, then inspect the corresponding USLaunchReport video frames and source-reduced camera audio.

The notebook writes provisional JSON products only.  It does not modify `project_config.py`.  The BCHH pick JSON includes a `source_event_times_config` mapping that is ready to copy into the project configuration after the events have been identified.


In [1]:
from __future__ import annotations

import json
import pickle
import subprocess
import sys
from pathlib import Path

import av
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display, clear_output
from obspy import Stream, UTCDateTime, read, Trace
from scipy.io import wavfile
from pyproj import Geod

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from modules import project_config as config
from modules.kml_utils import read_kml_points

if config.USLAUNCHREPORT_VIDEO_FILE is None:
    raise RuntimeError(
        'Set FALCON9_USLAUNCHREPORT_VIDEO_FILE to a locally obtained '
        'copy of the USLaunchReport source video.'
    )
VIDEO_FILE = config.USLAUNCHREPORT_VIDEO_FILE
ANALYSIS_CONFIG_FILE = config.NB010_DIR / 'analysis_configuration.json'
WEATHER_SUMMARY_FILE = config.NB020_DIR / 'weather_acoustic_summary.csv'
TRIAL_DIR = config.NB150_DIR
TRIAL_DIR.mkdir(parents=True, exist_ok=True)
FULL_AUDIO_WAV = TRIAL_DIR / 'uslaunchreport_audio_8khz_mono.wav'
TRIAL_JSON = TRIAL_DIR / 'alignment_trial.json'
CONFIRMED_EVENTS_JSON = TRIAL_DIR / 'uslaunchreport_video_event_picks.json'
BCHH_PICKS_JSON = TRIAL_DIR / 'bchh_strong_event_arrival_picks.json'

for required in (
    VIDEO_FILE,
    ANALYSIS_CONFIG_FILE,
    WEATHER_SUMMARY_FILE,
    config.KML_FILE,
):
    if not required.exists():
        raise FileNotFoundError(required)

analysis_config = json.loads(ANALYSIS_CONFIG_FILE.read_text())
weather_summary = pd.read_csv(WEATHER_SUMMARY_FILE).iloc[0]
BASELINE_STREAM_FILE = (
    config.NB010_DIR
    / 'bchh_corrected_moving_median_baseline_removed.pkl'
)
if not BASELINE_STREAM_FILE.is_file():
    raise FileNotFoundError(BASELINE_STREAM_FILE)

kml_points = read_kml_points(config.KML_FILE)
for required_point in ('SLC40', 'YouTube Video'):
    if required_point not in kml_points:
        raise KeyError(
            f'{required_point!r} is absent from {config.KML_FILE}; '
            f'available={sorted(kml_points)}'
        )
source_point = kml_points['SLC40']
camera_point = kml_points['YouTube Video']
camera_azimuth_deg, _, CAMERA_DISTANCE_M = Geod(ellps='WGS84').inv(
    float(source_point['lon']),
    float(source_point['lat']),
    float(camera_point['lon']),
    float(camera_point['lat']),
)
camera_azimuth_deg %= 360.0
still_air_speed_mps = float(weather_summary['still_air_sound_speed_mps'])
wind_speed_mps = float(
    weather_summary['path_vector_mean_wind_speed_mps']
)
wind_from_deg = float(
    weather_summary['path_vector_mean_wind_direction_from_deg']
)
wind_to_deg = (wind_from_deg + 180.0) % 360.0
camera_wind_along_ray_mps = wind_speed_mps * np.cos(
    np.deg2rad(camera_azimuth_deg - wind_to_deg)
)
CAMERA_SPEED_MPS = still_air_speed_mps + camera_wind_along_ray_mps
SOURCE_TO_BCHH_SPEED_MPS = float(
    weather_summary['effective_sound_speed_mps']
)
st_all = read(str(BASELINE_STREAM_FILE), format='PICKLE')
pressure_channels = tuple(analysis_config['pressure_channels'])
st_pressure = Stream(
    tr.copy() for tr in st_all if tr.stats.channel in pressure_channels
)
if len(st_pressure) != len(pressure_channels):
    raise ValueError(f'Expected {pressure_channels}; got {[tr.id for tr in st_pressure]}')

if not FULL_AUDIO_WAV.exists():
    command = [
        'ffmpeg', '-y', '-loglevel', 'error', '-i', str(VIDEO_FILE),
        '-vn', '-ac', '1', '-ar', '8000', '-c:a', 'pcm_s16le',
        str(FULL_AUDIO_WAV),
    ]
    subprocess.run(command, check=True)
audio_rate_hz, audio_raw = wavfile.read(FULL_AUDIO_WAV)
audio_raw = np.asarray(audio_raw, dtype=float)
audio_raw -= np.nanmedian(audio_raw)
audio_scale = np.nanpercentile(np.abs(audio_raw), 99.8) or 1.0
audio_raw /= audio_scale
audio_pts_s = np.arange(audio_raw.size) / float(audio_rate_hz)
audio_tr = Trace(data=audio_raw.copy())
audio_tr.stats.sampling_rate = audio_rate_hz
audio_tr.filter('highpass', freq=300.0, corners=4, zerophase=True)
audio_filtered = audio_tr.data

print('Configuration:', config.__file__)
print('Video:', VIDEO_FILE)
print('BCHH stream:', BASELINE_STREAM_FILE)
print('Channels:', [tr.stats.channel for tr in st_pressure])
print(
    'USLaunchReport camera propagation:',
    f'{CAMERA_DISTANCE_M:.1f} m / {CAMERA_SPEED_MPS:.2f} m/s = '
    f'{CAMERA_DISTANCE_M / CAMERA_SPEED_MPS:.3f} s',
)


Configuration: /Users/thompsong/Developer/KSCRocketSeismology/08_fireball_paper/falcon9-seismoacoustic-workflow-1.0.0/modules/project_config.py
Video: /Users/thompsong/Library/CloudStorage/Box-Box/thompsong/3_Project_Documents/NASAprojects/201602_Rocket_Seismology/02_KSC/spaceX_explosion_movie/SpaceX - Static Fire Anomaly - AMOS-6 - 09-01-2016.mov
BCHH stream: /Users/thompsong/Developer/KSCRocketSeismology/08_fireball_paper/falcon9-seismoacoustic-workflow-1.0.0/data/outputs/010_prepare_analysis_inputs/bchh_corrected_moving_median_baseline_removed.pkl
Channels: ['DD1', 'DD2', 'DD3']
USLaunchReport camera propagation: 4254.5 m / 349.80 m/s = 12.163 s


In [2]:
class VideoFrameReader:
    def __init__(self, path: Path):
        self.container = av.open(str(path))
        self.stream = self.container.streams.video[0]
        self.stream.thread_type = 'AUTO'

    def frame_at_pts(self, requested_s: float):
        seek_pts = int(max(0.0, requested_s) / float(self.stream.time_base))
        self.container.seek(seek_pts, stream=self.stream, backward=True)
        best = None
        best_s = np.nan
        best_error = np.inf
        for frame in self.container.decode(self.stream):
            if frame.pts is None:
                continue
            seconds = float(frame.pts * frame.time_base)
            error = abs(seconds - requested_s)
            if error < best_error:
                best, best_s, best_error = frame, seconds, error
            if seconds >= requested_s:
                break
        if best is None:
            return None, np.nan
        return best.to_ndarray(format='rgb24'), best_s

video_reader = VideoFrameReader(VIDEO_FILE)

def trace_distance_m(trace):
    geometry = getattr(trace.stats, 'source_geometry', None)
    if geometry is None:
        raise ValueError(f'{trace.id} lacks stats.source_geometry')
    return float(geometry.distance_m)

def robust_scale(values, percentile=99.5):
    value = np.nanpercentile(np.abs(np.asarray(values, float)), percentile)
    return float(value) if np.isfinite(value) and value > 0 else 1.0



## 1. Find the strongest catalogue events

This workbench reads the **full measured-event table written by Notebook 100**:

`100_measure_event_amplitudes_and_acoustic_seismic_coupling/event_acoustic_seismic_amplitudes.csv`

It ranks events by `acoustic_stack_peak_to_peak_pa`. Crucially, Notebook 100's `reduced_event_epoch_s` is an **array-reduced acoustic arrival at the BCHH reference channel**, not a source time. Notebook 150 therefore subtracts the full source-to-reference-channel travel time before selecting video or source-reducing any waveform. The dropdown contains the approximately 10 strongest measured events as well as configured named events not already represented.

After that conversion, all BCHH waveforms are plotted after subtracting their individual source-to-sensor acoustic travel times, so their onsets should fall close to the common source time (`t=0`).


In [3]:
# ---- Authoritative Notebook 100 amplitude catalogue ------------------------
N_STRONGEST = 10
CATALOG_FILE = (
    config.NB100_DIR
    / 'event_acoustic_seismic_amplitudes.csv'
)
TIME_COLUMN = 'reduced_event_epoch_s'
STRENGTH_COLUMN = 'acoustic_stack_peak_to_peak_pa'
CATALOG_REFERENCE_CHANNEL = analysis_config['infrasound_reference_channel']
reference_matches = [
    tr for tr in st_pressure if tr.stats.channel == CATALOG_REFERENCE_CHANNEL
]
if len(reference_matches) != 1:
    raise ValueError(
        f'Expected one catalogue reference channel {CATALOG_REFERENCE_CHANNEL}; '
        f'found {[tr.stats.channel for tr in st_pressure]}'
    )
CATALOG_REFERENCE_DISTANCE_M = trace_distance_m(reference_matches[0])

if not CATALOG_FILE.exists():
    raise FileNotFoundError(
        f'{CATALOG_FILE}\n'
        'Run Notebook 100 (event amplitudes and acoustic–seismic coupling) first. '
        'Notebook 150 deliberately uses its full measured-event table rather than the '
        'four named events in project_config.'
    )

catalog = pd.read_csv(CATALOG_FILE)
required_columns = {'event_number', TIME_COLUMN, STRENGTH_COLUMN}
missing = required_columns.difference(catalog.columns)
if missing:
    raise KeyError(
        f'Notebook 100 table is missing required columns {sorted(missing)}. '
        f'Available columns: {list(catalog.columns)}'
    )

working = catalog.copy()
working['_strength'] = pd.to_numeric(
    working[STRENGTH_COLUMN], errors='coerce'
)
working['_epoch'] = pd.to_numeric(working[TIME_COLUMN], errors='coerce')
working = working.loc[
    np.isfinite(working['_strength']) & np.isfinite(working['_epoch'])
].copy()
working = working.sort_values('_strength', ascending=False).head(N_STRONGEST)
working = working.reset_index().rename(columns={'index': '_catalog_row_index'})

STRONG_EVENTS = []
EVENT_META = {}
for rank, row in working.iterrows():
    event_number = int(row['event_number'])
    event_id = f'event_{event_number:03d}'
    reference_arrival_utc = UTCDateTime(float(row['_epoch']))
    provisional_source_utc = (
        reference_arrival_utc
        - CATALOG_REFERENCE_DISTANCE_M / SOURCE_TO_BCHH_SPEED_MPS
    )
    label = (
        f'#{rank + 1:02d} strongest — event {event_number:03d} — '
        f'arr {reference_arrival_utc.strftime("%H:%M:%S.%f")[:-3]} — '
        f'{float(row[STRENGTH_COLUMN]):.1f} Pa p2p'
    )
    meta = {
        'event_id': event_id,
        'event_number': event_number,
        'rank': rank + 1,
        'catalog_row_index': int(row['_catalog_row_index']),
        'reference_arrival_utc': reference_arrival_utc,
        'reference_channel': CATALOG_REFERENCE_CHANNEL,
        'reference_distance_m': CATALOG_REFERENCE_DISTANCE_M,
        'provisional_source_utc_meteorological': provisional_source_utc,
        'strength': float(row[STRENGTH_COLUMN]),
        'display_label': label,
    }
    STRONG_EVENTS.append(meta)
    EVENT_META[event_id] = meta

if not STRONG_EVENTS:
    raise ValueError(f'No finite measured acoustic amplitudes found in {CATALOG_FILE}')

# Freeze the measured subset before appending any named-only events below.
# Deduplication must compare named events only against these measured top-10
# records, all of which have a provisional source-time estimate.
MEASURED_STRONG_EVENTS = list(STRONG_EVENTS)

# Always retain the four configured named events. If one already
# coincides with a top-10 measured event, annotate that event instead of adding
# a duplicate. Otherwise add the configured named source time explicitly; this
# also retains events (e.g. a weak/seismic-dominant impact) that may not have a
# useful acoustic-amplitude row in Notebook 100.
NAMED_EVENTS = {
    'Upper stage': 'upper_stage',
    'Lower stage': 'lower_stage',
    'Payload impact': 'capsule_impact',
    'Payload explosion': 'capsule_explosion',
}
NAMED_DEDUP_TOLERANCE_S = 0.25

for named_label, config_key in NAMED_EVENTS.items():
    if config_key not in config.SOURCE_EVENT_TIMES:
        print(f'WARNING: named event {config_key!r} is absent from SOURCE_EVENT_TIMES')
        continue
    named_source = UTCDateTime(config.SOURCE_EVENT_TIMES[config_key])
    nearest = min(
        MEASURED_STRONG_EVENTS,
        key=lambda m: abs(
            float(m['provisional_source_utc_meteorological'] - named_source)
        ),
    )
    difference = float(
        nearest['provisional_source_utc_meteorological'] - named_source
    )
    if abs(difference) <= NAMED_DEDUP_TOLERANCE_S:
        nearest.setdefault('named_labels', []).append(named_label)
        nearest['display_label'] += f' — [{named_label}]'
        nearest['configured_named_source_utc'] = named_source
        print(
            f'Named event {named_label!r} matches top-10 {nearest["event_id"]} '
            f'within {difference:+.3f} s'
        )
    else:
        event_id = f'named_{config_key}'
        meta = {
            'event_id': event_id,
            'event_number': None,
            'rank': None,
            'catalog_row_index': None,
            'reference_arrival_utc': None,
            'reference_channel': None,
            'reference_distance_m': None,
            'fixed_source_utc': named_source,
            'strength': np.nan,
            'named_labels': [named_label],
            'display_label': (
                f'Named — {named_label} — source '
                f'{named_source.strftime("%H:%M:%S.%f")[:-3]}'
            ),
        }
        STRONG_EVENTS.append(meta)
        EVENT_META[event_id] = meta
        print(f'Added configured named event {named_label!r}')

EVENT_DROPDOWN_OPTIONS = [
    (meta['display_label'], meta['event_id']) for meta in STRONG_EVENTS
]

print('Notebook 100 measured-event table:', CATALOG_FILE)
print('Measured events in table:', len(catalog))
print('Catalogue epoch convention: arrival reduced to reference channel, NOT source time')
print('Reference channel:', CATALOG_REFERENCE_CHANNEL,
      f'({CATALOG_REFERENCE_DISTANCE_M:.1f} m from source)')
print('Finite acoustic amplitudes:', int(np.isfinite(pd.to_numeric(
    catalog[STRENGTH_COLUMN], errors='coerce')).sum()))
print(f'Showing the {len(STRONG_EVENTS)} strongest by {STRENGTH_COLUMN}:')
display(pd.DataFrame([{
    'rank': meta['rank'],
    'event_number': meta['event_number'],
    'name': ', '.join(meta.get('named_labels', [])),
    'reference_arrival_utc': (
        str(meta['reference_arrival_utc']) if meta.get('reference_arrival_utc') else None
    ),
    'source_utc': str(
        meta.get(
            'fixed_source_utc',
            meta.get('provisional_source_utc_meteorological'),
        )
    ),
    'acoustic_stack_peak_to_peak_pa': meta['strength'],
} for meta in STRONG_EVENTS]))


Added configured named event 'Upper stage'
Named event 'Lower stage' matches top-10 event_014 within -0.026 s
Named event 'Payload impact' matches top-10 event_022 within -0.027 s
Named event 'Payload explosion' matches top-10 event_023 within -0.014 s
Notebook 100 measured-event table: /Users/thompsong/Developer/KSCRocketSeismology/08_fireball_paper/falcon9-seismoacoustic-workflow-1.0.0/data/outputs/100_measure_event_amplitudes_and_acoustic_seismic_coupling/event_acoustic_seismic_amplitudes.csv
Measured events in table: 153
Catalogue epoch convention: arrival reduced to reference channel, NOT source time
Reference channel: DD2 (1408.3 m from source)
Finite acoustic amplitudes: 153
Showing the 11 strongest by acoustic_stack_peak_to_peak_pa:


,rank,event_number,name,reference_arrival_utc,source_utc,acoustic_stack_peak_to_peak_pa
0,1.0,14.0,Lower stage,2016-09-01T13:07:19.500134Z,2016-09-01T13:07:15.487510Z,1465.790874
1,2.0,13.0,,2016-09-01T13:07:19.427133Z,2016-09-01T13:07:15.414509Z,1465.790874
2,3.0,15.0,,2016-09-01T13:07:19.583134Z,2016-09-01T13:07:15.570510Z,423.882286
3,4.0,23.0,Payload explosion,2016-09-01T13:07:28.985893Z,2016-09-01T13:07:24.973269Z,404.676684
4,5.0,22.0,Payload impact,2016-09-01T13:07:28.405903Z,2016-09-01T13:07:24.393279Z,378.538112
5,6.0,24.0,,2016-09-01T13:07:30.451133Z,2016-09-01T13:07:26.438509Z,338.881115
6,7.0,12.0,,2016-09-01T13:07:19.239567Z,2016-09-01T13:07:15.226943Z,292.396181
7,8.0,54.0,,2016-09-01T13:14:17.363133Z,2016-09-01T13:14:13.350509Z,268.643079
8,9.0,18.0,,2016-09-01T13:07:23.467133Z,2016-09-01T13:07:19.454509Z,238.872237
9,10.0,25.0,,2016-09-01T13:07:32.797893Z,2016-09-01T13:07:28.785269Z,196.329568


In [4]:
BCHH_PICK_SPEED_MPS = SOURCE_TO_BCHH_SPEED_MPS
TIMING_CONVENTION = 'catalog_reference_arrival_minus_meteorological_travel_v3'
BCHH_EVENT_PICKS = {}
if BCHH_PICKS_JSON.exists():
    try:
        previous_collection = json.loads(BCHH_PICKS_JSON.read_text())
        if previous_collection.get('timing_convention') == TIMING_CONVENTION:
            BCHH_EVENT_PICKS.update(previous_collection.get('picks', {}))
        elif previous_collection.get('picks'):
            print('Ignoring picks made with an older timing convention.')
    except Exception as exc:
        print('Could not reload previous BCHH picks:', exc)

pick_offset_widgets = {
    tr.stats.channel: widgets.FloatSlider(
        value=0.0, min=-1.5, max=1.5, step=0.001,
        description=f'{tr.stats.channel} pick Δs', continuous_update=False,
        readout_format='.3f', style={'description_width': '120px'},
        layout=widgets.Layout(width='470px'),
    )
    for tr in st_pressure
}
audio_pick_widget = widgets.FloatSlider(
    value=0.0, min=-1.5, max=1.5, step=0.001,
    description='Audio pick Δs', continuous_update=False,
    readout_format='.3f', style={'description_width': '120px'},
    layout=widgets.Layout(width='470px'),
)
save_bchh_picks_button = widgets.Button(
    description='Save precise BCHH + audio picks', button_style='success'
)
pick_save_output = widgets.Output()

def selected_source_utc(event_id):
    if event_id in BCHH_EVENT_PICKS:
        return UTCDateTime(BCHH_EVENT_PICKS[event_id]['median_source_utc'])
    meta = EVENT_META[event_id]
    if meta.get('fixed_source_utc') is not None:
        return UTCDateTime(meta['fixed_source_utc'])
    speed = (
        float(speed_widget.value)
        if 'speed_widget' in globals()
        else BCHH_PICK_SPEED_MPS
    )
    # Notebook 100's reduced_event_epoch_s is the acoustic arrival reduced
    # across the array to its reference channel. It is NOT a source epoch.
    return (
        meta['reference_arrival_utc']
        - meta['reference_distance_m'] / speed
    )

def load_saved_offsets(event_id):
    record = BCHH_EVENT_PICKS.get(event_id, {})
    reference_utc = selected_source_utc(event_id)
    for channel, widget in pick_offset_widgets.items():
        channel_record = record.get('channels', {}).get(channel, {})
        if 'source_utc' in channel_record:
            value = float(UTCDateTime(channel_record['source_utc']) - reference_utc)
        else:
            value = 0.0
        widget.value = max(widget.min, min(widget.max, value))
    audio_record = record.get('camera_audio', {})
    if 'source_utc' in audio_record:
        audio_value = float(UTCDateTime(audio_record['source_utc']) - reference_utc)
    else:
        audio_value = 0.0
    audio_pick_widget.value = max(
        audio_pick_widget.min, min(audio_pick_widget.max, audio_value)
    )

def save_bchh_picks(_):
    event_id = event_widget.value
    meta = EVENT_META[event_id]
    catalog_reference_arrival_utc = meta.get('reference_arrival_utc')
    reference_utc = selected_source_utc(event_id)
    speed = float(speed_widget.value)
    channels = {}
    source_epochs = []
    for tr in st_pressure:
        channel = tr.stats.channel
        offset = float(pick_offset_widgets[channel].value)
        travel = trace_distance_m(tr) / speed
        source = reference_utc + offset
        arrival = source + travel
        source_epochs.append(float(source.timestamp))
        channels[channel] = {
            'arrival_utc': str(arrival),
            'reduced_time_offset_from_reference_s': offset,
            'distance_m': trace_distance_m(tr),
            'travel_time_s': travel,
            'source_utc': str(source),
        }
    median_source = UTCDateTime(float(np.median(source_epochs)))
    residuals = np.asarray(source_epochs) - float(median_source.timestamp)
    audio_pick_offset = float(audio_pick_widget.value)
    audio_source = reference_utc + audio_pick_offset
    camera_travel = CAMERA_DISTANCE_M / float(camera_speed_widget.value)
    # Invert the same reduced-time equation used by the plotted audio trace.
    audio_pts = (
        audio_pick_offset
        + config.VIDEO_ANCHOR_MOVIE_S
        + video_anchor_adjustment_widget.value
        + float(reference_utc - config.VIDEO_ANCHOR_UTC)
        + camera_travel
        + audio_adjustment_widget.value
    )
    BCHH_EVENT_PICKS[event_id] = {
        'event_id': event_id,
        'rank': meta.get('rank'),
        'catalog_row_index': meta.get('catalog_row_index'),
        'catalog_reference_arrival_utc': (
            str(catalog_reference_arrival_utc)
            if catalog_reference_arrival_utc is not None else None
        ),
        'catalog_reference_channel': meta.get('reference_channel'),
        'catalog_reference_distance_m': meta.get('reference_distance_m'),
        'pick_reference_source_utc': str(reference_utc),
        'catalog_strength': (
            float(meta['strength'])
            if np.isfinite(meta.get('strength', np.nan)) else None
        ),
        'named_labels': meta.get('named_labels', []),
        'bchh_speed_mps': speed,
        'channels': channels,
        'median_source_utc': str(median_source),
        'source_time_channel_residuals_s': {
            tr.stats.channel: float(residual)
            for tr, residual in zip(st_pressure, residuals)
        },
        'source_time_peak_to_peak_scatter_s': float(np.ptp(source_epochs)),
        'camera_audio': {
            'reduced_time_pick_s': audio_pick_offset,
            'source_utc': str(audio_source),
            'audio_pts_s': float(audio_pts),
            'camera_distance_m': CAMERA_DISTANCE_M,
            'camera_speed_mps': float(camera_speed_widget.value),
            'camera_travel_time_s': float(camera_travel),
            'audio_residual_s': float(audio_adjustment_widget.value),
            'video_anchor_adjustment_s': float(video_anchor_adjustment_widget.value),
            'note': 'Audio source UTC is conditional on the trial USLaunchReport video/audio anchor; BCHH remains the reference clock.',
        },
    }
    collection = {
        'status': 'manual_bchh_arrival_picks',
        'producer_notebook': '150_align_uslaunchreport_video.ipynb',
        'reference_clock': 'BCHH',
        'timing_convention': TIMING_CONVENTION,
        'catalog_file': str(CATALOG_FILE),
        'picks': BCHH_EVENT_PICKS,
        'source_event_times_config': {
            key: value['median_source_utc']
            for key, value in BCHH_EVENT_PICKS.items()
        },
    }
    BCHH_PICKS_JSON.write_text(json.dumps(collection, indent=2) + '\n')
    with pick_save_output:
        clear_output(wait=True)
        print('Saved:', BCHH_PICKS_JSON)
        print('Median BCHH-derived source UTC:', median_source)
        print('Channel residuals (s):',
              BCHH_EVENT_PICKS[event_id]['source_time_channel_residuals_s'])
        print('Camera-audio reduced-time pick:', f'{audio_pick_offset:+.3f} s')
        print('Camera-audio PTS:', f'{audio_pts:.6f} s')
        print(f"Config-ready: {event_id!r}: UTCDateTime({str(median_source)!r}),")
    # Re-express the three channel picks as residuals about their new median.
    load_saved_offsets(event_id)
    render_alignment()


## 3. Align USLaunchReport video/audio to the BCHH-derived event times

This is the main workbench. Select one of the strongest catalog events and inspect the **video frames, 300-Hz-high-pass camera audio, and raw travel-time-reduced DD1–DD3 together**. The three `pick Δs` sliders mark precise BCHH onsets at 1-ms increments. Reduce `Half-window s` to zoom tightly around an onset.

The top filmstrip shows three native movie frames. The second panel shows camera audio after subtracting the pad-to-camera acoustic travel time. DD1–DD3 are **raw** apart from the time-axis shift by their individual \(d/c\). `Video anchor Δs` therefore measures the adjustment required to map the untimestamped USLaunchReport movie onto BCHH time.


In [5]:
event_widget = widgets.Dropdown(
    options=EVENT_DROPDOWN_OPTIONS, value=STRONG_EVENTS[0]['event_id'], description='Event'
)
video_anchor_adjustment_widget = widgets.FloatSlider(
    value=0.0, min=-2.0, max=2.0, step=1/config.VIDEO_NOMINAL_FPS,
    description='Video anchor Δs', continuous_update=False,
    readout_format='.3f', style={'description_width': '150px'},
)
frame_offset_widget = widgets.BoundedIntText(
    value=0, min=-300, max=300, description='Center frame Δ',
    style={'description_width': '150px'},
)
subframe_fraction_widget = widgets.FloatSlider(
    value=0.0, min=-0.5, max=0.5, step=0.05,
    description='Between-frame fraction', continuous_update=False,
    readout_format='.2f', style={'description_width': '150px'},
)
previous_frame_button = widgets.Button(description='← Previous frame')
next_frame_button = widgets.Button(description='Next frame →')
camera_speed_widget = widgets.FloatSlider(
    value=CAMERA_SPEED_MPS, min=300.0, max=450.0, step=0.5,
    description='Camera speed m/s', continuous_update=False,
    readout_format='.1f', style={'description_width': '150px'},
)
audio_adjustment_widget = widgets.FloatSlider(
    value=0.0, min=-2.0, max=2.0, step=0.01,
    description='Audio residual s', continuous_update=False,
    readout_format='.3f', style={'description_width': '150px'},
)
speed_widget = widgets.FloatSlider(
    value=BCHH_PICK_SPEED_MPS, min=300.0, max=450.0, step=0.5,
    description='BCHH speed m/s', continuous_update=False,
    readout_format='.1f', style={'description_width': '150px'},
)
bchh_residual_widget = widgets.FloatSlider(
    value=0.0, min=-2.0, max=2.0, step=0.01,
    description='BCHH residual s', continuous_update=False,
    readout_format='.3f', style={'description_width': '150px'},
)
event_nudge_widget = widgets.FloatSlider(
    value=0.0, min=-2.0, max=2.0, step=0.01, #step=1/config.VIDEO_NOMINAL_FPS,
    description='Event nudge s', continuous_update=False,
    readout_format='.3f', style={'description_width': '150px'},
)
half_window_widget = widgets.FloatSlider(
    value=2.0, min=0.05, max=8.0, step=0.05,
    description='Half-window s', continuous_update=False,
    readout_format='.2f', style={'description_width': '150px'},
)
filter_widget = widgets.Checkbox(value=True, description='0.5–20 Hz BCHH filter')
output = widgets.Output()

def render_alignment(*_):
    with output:
        clear_output(wait=True)
        event_key = event_widget.value
        source_utc = selected_source_utc(event_key)
        anchor_delta_s = video_anchor_adjustment_widget.value
        audio_residual_s = audio_adjustment_widget.value
        camera_speed_mps = camera_speed_widget.value
        camera_travel_s = CAMERA_DISTANCE_M / camera_speed_mps
        speed_mps = speed_widget.value
        bchh_residual_s = 0.0  # BCHH is the reference clock
        event_nudge_s = 0.0
        half_window_s = half_window_widget.value

        frame_period_s = 1.0 / config.VIDEO_NOMINAL_FPS
        event_video_pts_s = (
            config.VIDEO_ANCHOR_MOVIE_S
            + float(source_utc - config.VIDEO_ANCHOR_UTC)
            + anchor_delta_s
            + frame_offset_widget.value * frame_period_s
        )

        fig = plt.figure(figsize=(13, 9), constrained_layout=True)
        grid = fig.add_gridspec(5, 1, height_ratios=[3.2, 1, 1, 1, 1])
        video_grid = grid[0].subgridspec(1, 3, wspace=0.02)
        actual_frame_pts = []
        for panel_index, relative_frame in enumerate((-1, 0, 1)):
            requested_pts = event_video_pts_s + relative_frame * frame_period_s
            frame, actual_pts = video_reader.frame_at_pts(requested_pts)
            actual_frame_pts.append(actual_pts)
            ax_video = fig.add_subplot(video_grid[0, panel_index])
            if frame is not None:
                ax_video.imshow(frame)
            ax_video.set_axis_off()
            label = 'CENTER' if relative_frame == 0 else (
                'PREVIOUS' if relative_frame < 0 else 'NEXT'
            )
            ax_video.set_title(
                f'{label}: PTS {actual_pts:.6f} s\n'
                f'frame {frame_offset_widget.value + relative_frame:+d}',
                color='firebrick' if relative_frame == 0 else 'black',
                fontsize=9,
            )
        center_pts_s = actual_frame_pts[1]
        estimated_event_pts_s = (
            center_pts_s
            + subframe_fraction_widget.value * frame_period_s
        )
        estimated_event_utc = (
            config.VIDEO_ANCHOR_UTC
            + estimated_event_pts_s
            - config.VIDEO_ANCHOR_MOVIE_S
            - anchor_delta_s
        )
        fig.suptitle(
            f'USLaunchReport — {EVENT_META[event_key]["display_label"]}; estimated event PTS '
            f'{estimated_event_pts_s:.6f} s; mapped UTC {estimated_event_utc}',
            fontsize=11,
        )

        axes = [fig.add_subplot(grid[i]) for i in range(1, 5)]
        # Map every audio sample to UTC through the trial video anchor, then
        # reduce it by the physical pad-to-camera acoustic travel time. The
        # independent residual is applied only after that physical reduction.
        # Audio PTS always comes from the unmodified sample clock. The
        # optional filter changes amplitudes, never the time coordinate.
        audio_values = audio_filtered
        audio_relative_s = (
            audio_pts_s - config.VIDEO_ANCHOR_MOVIE_S - anchor_delta_s
            - float(source_utc - config.VIDEO_ANCHOR_UTC)
            - camera_travel_s
            - audio_residual_s
        )
        mask = np.abs(audio_relative_s) <= half_window_s
        stride = max(1, int(mask.sum() / 12000))
        axes[0].plot(audio_relative_s[mask][::stride], audio_values[mask][::stride],
                     color='0.2', lw=0.55)
        axes[0].axvline(audio_pick_widget.value, color='tab:green', ls='--',
                           lw=1.2, label='precise camera-audio pick')
        axes[0].set_ylabel('USLaunchReport audio')
        axes[0].set_title(
            f'Source-reduced camera audio: subtract {camera_travel_s:.3f} s '
            f'({CAMERA_DISTANCE_M:.1f} m / {camera_speed_mps:.1f} m/s) '
            f'and residual {audio_residual_s:+.3f} s',
            fontsize=9,
        )

        # Raw BCHH infrasound: no detrend, taper, or frequency filter.
        st_plot = st_pressure.copy()
        for ax, trace in zip(axes[1:], st_plot):
            travel_s = trace_distance_m(trace) / speed_mps
            start = source_utc + travel_s - half_window_s + bchh_residual_s
            end = source_utc + travel_s + half_window_s + bchh_residual_s
            tr = trace.copy().trim(start, end, pad=True, fill_value=0)
            observed_relative_s = tr.times(reftime=source_utc)
            reduced_relative_s = observed_relative_s - travel_s - bchh_residual_s
            values = tr.data.astype(float) / robust_scale(tr.data)
            ax.plot(reduced_relative_s, values, color='black', lw=0.7)
            pick_offset = pick_offset_widgets[trace.stats.channel].value
            ax.axvline(pick_offset, color='tab:blue', ls='--', lw=1.2,
                       label=f'{trace.stats.channel} precise pick')
            ax.set_ylabel(trace.stats.channel)
            ax.set_title(
                f'Source-reduced {trace.stats.channel}: subtract '
                f'{travel_s:.3f} s travel and residual '
                f'{bchh_residual_s:+.3f} s', fontsize=8, loc='left'
            )
            ax.text(0.995, 0.88, f'd={trace_distance_m(trace):.1f} m; '
                    f'travel={travel_s:.3f} s', ha='right', va='top',
                    transform=ax.transAxes, fontsize=8)

        for ax in axes:
            ax.axvline(0.0, color='firebrick', lw=1.1, label='BCHH/catalog source reference')
            ax.set_xlim(-half_window_s, half_window_s)
            ax.grid(True, alpha=0.16)
        axes[-1].set_xlabel('Trial source-aligned time relative to event (s)')
        axes[0].legend(loc='upper right', fontsize=8)
        plt.show()
        print(
            f'video anchor adjustment={anchor_delta_s:+.3f} s; '
            f'center frame={frame_offset_widget.value:+d}; '
            f'between-frame fraction={subframe_fraction_widget.value:+.2f}; '
            f'camera travel={camera_travel_s:.3f} s; '
            f'audio residual={audio_residual_s:+.3f} s; '
            f'BCHH speed={speed_mps:.1f} m/s; '
            f'BCHH residual={bchh_residual_s:+.3f} s; '
            f'event nudge={event_nudge_s:+.3f} s'
        )

controls = [event_widget, video_anchor_adjustment_widget, frame_offset_widget,
            subframe_fraction_widget, camera_speed_widget,
            audio_adjustment_widget, speed_widget, half_window_widget]
for control in controls:
    control.observe(render_alignment, names='value')
for pick_widget in pick_offset_widgets.values():
    pick_widget.observe(render_alignment, names='value')
audio_pick_widget.observe(render_alignment, names='value')
def event_changed_for_picks(change):
    load_saved_offsets(change['new'])
event_widget.observe(event_changed_for_picks, names='value')
def previous_frame(_):
    frame_offset_widget.value -= 1

def next_frame(_):
    frame_offset_widget.value += 1

previous_frame_button.on_click(previous_frame)
next_frame_button.on_click(next_frame)
navigation = widgets.HBox([previous_frame_button, next_frame_button])
load_saved_offsets(event_widget.value)
display(widgets.VBox([
    widgets.HBox([widgets.VBox(controls[:6]), widgets.VBox(controls[6:])]),
    navigation,
    widgets.HTML('<b>Precise reduced-time picks (1 ms increments)</b>'),
    audio_pick_widget,
    *pick_offset_widgets.values(),
    save_bchh_picks_button,
    pick_save_output,
    output,
]))
render_alignment()


## 4. Save provisional video identifications

These records preserve the chosen movie frame/PTS and alignment controls for each BCHH-ranked event.  They remain provisional until the physical event visible in the footage has been identified and the common USLaunchReport↔BCHH anchor has been chosen.


In [6]:
save_button = widgets.Button(description='Save provisional trial', button_style='warning')
confirm_frame_button = widgets.Button(
    description='Confirm video event time', button_style='success'
)
frame_confidence_widget = widgets.Dropdown(
    options=['high', 'medium', 'low'], value='high', description='Frame confidence'
)
frame_notes_widget = widgets.Text(
    description='Video notes', placeholder='What begins in this frame?'
)
save_output = widgets.Output()

def save_trial(_):
    event_key = event_widget.value
    trial = {
        'status': 'provisional_alignment_trial',
        'producer_notebook': '150_align_uslaunchreport_video.ipynb',
        'project_config': str(config.__file__),
        'video_file': str(VIDEO_FILE),
        'event_id': event_key,
        'event_source_utc': str(selected_source_utc(event_key)),
        'configured_video_anchor_movie_s': config.VIDEO_ANCHOR_MOVIE_S,
        'configured_video_anchor_utc': str(config.VIDEO_ANCHOR_UTC),
        'video_anchor_adjustment_s': video_anchor_adjustment_widget.value,
        'center_frame_offset': frame_offset_widget.value,
        'between_frame_fraction': subframe_fraction_widget.value,
        'uslaunchreport_camera_distance_m': CAMERA_DISTANCE_M,
        'uslaunchreport_camera_speed_mps': camera_speed_widget.value,
        'uslaunchreport_camera_travel_time_s': (
            CAMERA_DISTANCE_M / camera_speed_widget.value
        ),
        'uslaunchreport_audio_residual_s': audio_adjustment_widget.value,
        'bchh_acoustic_speed_mps': speed_widget.value,
        'bchh_reduced_time_residual_s': bchh_residual_widget.value,
        'event_specific_nudge_s': event_nudge_widget.value,
        'display_half_window_s': half_window_widget.value,
        'pressure_channels': list(pressure_channels),
        'waveform_product': str(BASELINE_STREAM_FILE),
    }
    TRIAL_JSON.write_text(json.dumps(trial, indent=2) + '\n')
    with save_output:
        clear_output(wait=True)
        print('Saved provisional trial:', TRIAL_JSON)
        print(json.dumps(trial, indent=2))

def confirm_video_event_time(_):
    event_key = event_widget.value
    source_utc = selected_source_utc(event_key)
    anchor_delta_s = video_anchor_adjustment_widget.value
    frame_period_s = 1.0 / config.VIDEO_NOMINAL_FPS
    requested_center_pts_s = (
        config.VIDEO_ANCHOR_MOVIE_S
        + float(source_utc - config.VIDEO_ANCHOR_UTC)
        + anchor_delta_s
        + frame_offset_widget.value * frame_period_s
    )
    _, actual_center_pts_s = video_reader.frame_at_pts(requested_center_pts_s)
    estimated_pts_s = (
        actual_center_pts_s
        + subframe_fraction_widget.value * frame_period_s
    )
    estimated_utc = (
        config.VIDEO_ANCHOR_UTC
        + estimated_pts_s
        - config.VIDEO_ANCHOR_MOVIE_S
        - anchor_delta_s
    )
    record = {
        'event_id': event_key,
        'event_label': event_widget.value,
        'status': 'manually_confirmed_video_pick',
        'actual_center_frame_pts_s': actual_center_pts_s,
        'center_frame_offset': frame_offset_widget.value,
        'between_frame_fraction': subframe_fraction_widget.value,
        'estimated_event_video_pts_s': estimated_pts_s,
        'estimated_event_utc': str(estimated_utc),
        'one_frame_duration_s': frame_period_s,
        'nominal_half_frame_uncertainty_s': frame_period_s / 2.0,
        'video_anchor_adjustment_s': anchor_delta_s,
        'bchh_derived_source_utc': str(source_utc),
        'difference_from_bchh_source_s': float(estimated_utc - source_utc),
        'confidence': frame_confidence_widget.value,
        'notes': frame_notes_widget.value,
        'video_file': str(VIDEO_FILE),
    }
    collection = (
        json.loads(CONFIRMED_EVENTS_JSON.read_text())
        if CONFIRMED_EVENTS_JSON.exists()
        else {
            'status': 'provisional_manual_video_event_picks',
            'producer_notebook': '150_align_uslaunchreport_video.ipynb',
            'picks': {},
        }
    )
    collection.setdefault('picks', {})[event_key] = record
    CONFIRMED_EVENTS_JSON.write_text(
        json.dumps(collection, indent=2) + '\n'
    )
    with save_output:
        clear_output(wait=True)
        print('Confirmed provisional video event time:', CONFIRMED_EVENTS_JSON)
        print(json.dumps(record, indent=2))

save_button.on_click(save_trial)
confirm_frame_button.on_click(confirm_video_event_time)
display(
    widgets.VBox([
        widgets.HBox([confirm_frame_button, save_button]),
        frame_confidence_widget,
        frame_notes_widget,
        save_output,
    ])
)


In [7]:
tshift = [-0.044, 0.023, 0.019, 0.071, 0.013]
print(np.mean(tshift), np.std(tshift))

0.016399999999999998 0.036581962768555756


In [8]:
0.17/12*349

4.944166666666667